# Clinicopathological and MSI Status Data for Second Primary Colorectal Cancer in Cancer Survivors, 2019-2025 Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.sphg-zzsp) using the [mlcroissant](https://github.com/mlcommons/croissant) library.

The dataset includes 77 records of cancer survivors diagnosed with second primary colorectal cancer, with fields capturing demographics, comorbidities, anatomical and pathological data, immuno-oncology markers (MSI status), and more.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.sphg-zzsp/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed.
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.sphg-zzsp/fair2.json'

# Load the Croissant dataset package metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List the available record sets, fields (columns), and their `@id`s from the dataset package.

In [ ]:
# List RecordSet @ids
print("Available RecordSets (by @id):")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"- {rs['@id']}")

# For each record set, print its fields/columns and their @ids
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    fields = rs.get('fields', [])
    if fields:
        print("  Fields (by @id):")
        for field in fields:
            print(f"    - {field['@id']} ({field.get('name', '')})")
    columns = rs.get('columns', [])
    if columns:
        print("  Columns (by @id):")
        for col in columns:
            print(f"    - {col['@id']} ({col.get('name', '')})")

## 3. Data Extraction
Load data from a specific RecordSet into a DataFrame for analysis. All entities are referenced by their `@id`.

*Tip:* Use the listing above to select the record set and field/column `@id`s.

In [ ]:
# Identify record set ids (from above), choose the main table for analysis.
# Let's find the main patient data table. For many biomedical datasets, the first RecordSet or one named with patient/data/cases is the main table.
# Here, we programmatically select all record sets and load each to a DataFrame.

dataframes = {}
import json

for rs in dataset.metadata.record_sets:
    record_set_id = rs['@id']
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:  # Only add non-empty
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")


# Inspect DataFrames and available columns (fields/columns @id)
if dataframes:
    # Pick the largest/main DataFrame for further analysis
    main_rs_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print("\nMost records are in RecordSet:", main_rs_id)
    print("Columns (@id):\n", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No records loaded. Check RecordSet definitions above.")

## 4. Exploratory Data Analysis (EDA)

Let's process the main dataset by selecting some numeric/categorical fields (referenced by their `@id`s) and perform:
- record filtering by value,
- field normalization,
- simple grouping.

> *Note: All field/column references use their `@id`.*

In [ ]:
# Select a numeric field by @id (adjust based on your data fields, here we guess typical field names)
df = dataframes[main_rs_id].copy()

# Try to find candidate numeric fields (@id often ends with field name)
numeric_candidates = [col for col in df.columns if "age" in col.lower() or df[col].dtype.kind in 'fi']
print("Numeric field candidates (@id):", numeric_candidates)

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # fall back to any field if none guessed
    numeric_field_id = df.columns[0]
print(f"Using field for numeric analysis: {numeric_field_id}")

# Convert field if not already numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records with field > some threshold (e.g., age > 50)
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Select a categorical/grouping field (guess from columns)
cat_candidates = [f for f in filtered_df.columns if filtered_df[f].dtype == 'object' and f != numeric_field_id]
if cat_candidates:
    group_field_id = cat_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
    print(f"Grouped data by {group_field_id} (showing mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization

Visualize key distributions. For example, plot histogram for the selected numeric field and bar plot for the group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (after filtering)
plt.figure(figsize=(6,4))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=10, color='skyblue')
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
plt.show()

# If group_field_id was selected, plot group means
if 'group_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y=f"mean_{numeric_field_id}", data=grouped_df, palette='viridis')
    plt.xticks(rotation=45)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- This exploration demonstrated programmatic loading, schema-guided referencing (by `@id`), and basic analysis of the FAIR^2 dataset via `mlcroissant`.
- Users can easily extend the notebook to work with the specific fields and analyses most relevant to their research, always referencing `@id`s for reproducibility aligned with the Croissant standard.
- For more advanced processing or clinical research, consult the dataset metadata for field definitions, and be mindful of dataset limitations as described by the authors.